# Evaluating Models

In the previous notebook we built a regression model and measured its R² on the same data it was trained on. The explainer warned us why that number is unreliable: a model that memorises the training data can score brilliantly on what it has already seen and then fail on anything new. The exam analogy applies directly. A student who memorises past papers gets 100% on those papers and fails the real exam.

This notebook puts that idea into practice. We will hold back part of the data, train on the rest, and measure how well the model performs on properties it has never seen. We will also see overfitting happen in front of us, using polynomial regression to push a model past the point of usefulness.

We will:

- Split data into **training** and **test** sets with `train_test_split`
- Calculate **MAE**, **RMSE**, and **R²** on the test set
- Create a **residual plot** and interpret what it shows
- Demonstrate **overfitting** using polynomial regression
- Assess **feature importance** from regression coefficients
- Make and justify a **final model recommendation**

In [ ]:
%pip install -q pandas matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures

sns.set_style("whitegrid")

In [ ]:
df = pd.read_csv("../data/property_sales.csv")

features = ["floor_area_sqm", "bedrooms", "bathrooms", "garden_sqm",
            "age_years", "distance_to_station_km", "distance_to_school_km"]
X = df[features]
y = df["sale_price"]

print(f"Total samples: {len(df)}")

---

## 1. Train/test split

The principle is simple: if we evaluate a model on the same data it learned from, we are measuring memorisation. To measure generalisation, we need data the model has never seen.

The standard approach is to hold back 20-30% of the data before building the model. The model trains on the larger portion and is then tested on the held-out portion. `train_test_split` from scikit-learn handles this in one line.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set:     {len(X_test)} samples")

The `random_state` parameter makes the split reproducible. Without it, we would get a different split each time we run the cell, making comparisons unreliable.

In [ ]:
# Fit the model on training data only
model = LinearRegression()
model.fit(X_train, y_train)

# Predict on both sets
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

---

## 2. Error metrics

R² tells us the proportion of variance explained, but it does not tell us the size of the errors in pounds. For that, we need metrics in the original units. Three are standard for regression:

| Metric | Formula | Interpretation |
|---|---|---|
| **MAE** (Mean Absolute Error) | Average of \|actual - predicted\| | Average prediction error in £. The easiest metric to explain to a non-technical audience. |
| **RMSE** (Root Mean Squared Error) | Square root of the average of (actual - predicted)² | Also in £, but penalises large errors more heavily. Useful when big misses matter more than small ones. |
| **R²** | 1 - (sum of squared errors / total variance) | Proportion of variance explained. Unitless, 0 to 1. |

We calculate all three on both the training set and the test set. Comparing the two reveals whether the model generalises.

In [ ]:
# Training set metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = root_mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

# Test set metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = root_mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"{'Metric':<8} {'Train':>12} {'Test':>12}")
print(f"{'-'*32}")
print(f"{'MAE':<8} {'£' + f'{train_mae:,.0f}':>12} {'£' + f'{test_mae:,.0f}':>12}")
print(f"{'RMSE':<8} {'£' + f'{train_rmse:,.0f}':>12} {'£' + f'{test_rmse:,.0f}':>12}")
print(f"{'R²':<8} {train_r2:>12.3f} {test_r2:>12.3f}")

If the training and test metrics are close, the model generalises well. If training performance is substantially better, the model is overfitting. This is the diagnostic the explainer described: the gap between training and test performance tells us how much the model has memorised versus learned.

The MAE gives the estate agency a direct answer: "on average, how far off are the valuations?" If MAE is £25,000, the agency can expect its automated estimates to miss by about that much on a typical property.

---

## 3. Residual plots

The error metrics give us averages, but averages can hide problems. A **residual** is the difference between the actual value and the predicted value for a single observation:

$$\text{residual} = y_{\text{actual}} - y_{\text{predicted}}$$

Plotting residuals against predicted values shows us *where* the model goes wrong, not just *how much*. In a well-behaved model, the residuals should scatter randomly around zero with no visible pattern. Any structure in the residuals is a sign that the model is systematically missing something.

In [ ]:
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs predicted values
axes[0].scatter(y_test_pred, residuals, alpha=0.3, s=10)
axes[0].axhline(y=0, color="red", linestyle="--", linewidth=1)
axes[0].set_xlabel("Predicted price (£)")
axes[0].set_ylabel("Residual (£)")
axes[0].set_title("Residuals vs predicted values")

# Distribution of residuals
axes[1].hist(residuals, bins=40, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Residual (£)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Distribution of residuals")

plt.tight_layout()
plt.show()

**What to look for:**

- **Random scatter around zero:** the model is capturing the main trends. This is what we want.
- **Funnel shape** (residuals widen as predictions increase): the model's accuracy varies with price level. This is common with property data and is the heteroscedasticity the explainer described.
- **Curved pattern:** the model is missing a non-linear relationship. A straight line is the wrong summary for this feature.
- **Clusters or gaps:** there may be subgroups in the data that the model treats as a single population.

---

## 4. Overfitting demo: polynomial regression

The explainer introduced overfitting as the most important failure mode in predictive modelling. Now we are going to watch it happen.

**Polynomial regression** creates new features from the original: $x$, $x^2$, $x^3$, ..., $x^d$. Higher degrees give the model more flexibility to bend and twist through the data. At low degrees, this flexibility captures genuine curvature. At high degrees, the model starts chasing noise, fitting random variation that will not repeat in new data.

In [ ]:
# Use a single feature for clarity
X_floor = df[["floor_area_sqm"]]
y_price = df["sale_price"]

X_floor_train, X_floor_test, y_floor_train, y_floor_test = train_test_split(
    X_floor, y_price, test_size=0.2, random_state=42
)

degrees = [1, 2, 3, 5, 10, 15]
results = []

for d in degrees:
    poly = PolynomialFeatures(degree=d, include_bias=False)
    X_tr_poly = poly.fit_transform(X_floor_train)
    X_te_poly = poly.transform(X_floor_test)

    lr = LinearRegression()
    lr.fit(X_tr_poly, y_floor_train)

    train_r2 = lr.score(X_tr_poly, y_floor_train)
    test_r2 = lr.score(X_te_poly, y_floor_test)

    results.append({"degree": d, "train_r2": train_r2, "test_r2": test_r2})

results_df = pd.DataFrame(results)
results_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results_df["degree"], results_df["train_r2"], "o-", label="Training R²")
plt.plot(results_df["degree"], results_df["test_r2"], "o-", label="Test R²")
plt.xlabel("Polynomial degree")
plt.ylabel("R²")
plt.title("Overfitting: training R² vs test R²")
plt.legend()
plt.xticks(degrees)
plt.tight_layout()
plt.show()

**Key observations:**

- Training R² increases steadily with polynomial degree. The model fits the training data more closely at every step.
- Test R² improves at first (capturing genuine non-linearity) then levels off or drops. The model is fitting noise that does not appear in new data.
- The degree where test R² peaks is approximately the right level of complexity. Beyond that point, the model is memorising rather than learning.

This is the bias-variance trade-off from the explainer, made visible. Too simple (degree 1) misses curvature. Too complex (degree 15) chases noise. The best model sits in between.

### Visualising the polynomial fits

The numbers tell the story, but seeing the fitted curves makes overfitting unmistakable. At high degrees, the curve oscillates wildly between data points, contorting itself to pass through training observations that are just noise.

In [ ]:
x_plot = np.linspace(X_floor["floor_area_sqm"].min(), X_floor["floor_area_sqm"].max(), 300).reshape(-1, 1)

fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharey=True)

for ax, d in zip(axes.ravel(), degrees):
    poly = PolynomialFeatures(degree=d, include_bias=False)
    X_tr_poly = poly.fit_transform(X_floor_train)
    lr = LinearRegression().fit(X_tr_poly, y_floor_train)

    y_plot = lr.predict(poly.transform(x_plot))

    ax.scatter(X_floor_train, y_floor_train, alpha=0.15, s=5)
    ax.plot(x_plot, y_plot, color="red", linewidth=2)
    test_r2 = results_df.loc[results_df["degree"] == d, "test_r2"].values[0]
    ax.set_title(f"Degree {d} (test R² = {test_r2:.3f})")
    ax.set_xlabel("Floor area (sqm)")

axes[0, 0].set_ylabel("Sale price (£)")
axes[1, 0].set_ylabel("Sale price (£)")
plt.tight_layout()
plt.show()

Which features matter most for the agency's valuations? In a linear regression, the coefficient tells us how much each feature contributes to the prediction. But raw coefficients are not directly comparable because the features have different scales: floor area ranges from roughly 30 to 250, while bedrooms range from 1 to 6.

To put them on a comparable scale, multiply each coefficient by the standard deviation of its feature. This gives us the predicted price change for a one-standard-deviation shift in each feature.

In [ ]:
# Refit on all data for feature importance analysis
model_all = LinearRegression()
model_all.fit(X, y)

importance = pd.DataFrame({
    "feature": features,
    "coefficient": model_all.coef_,
    "std_dev": X.std().values,
})
importance["scaled_importance"] = abs(importance["coefficient"] * importance["std_dev"])
importance = importance.sort_values("scaled_importance", ascending=False)
importance

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(importance["feature"], importance["scaled_importance"])
plt.xlabel("Scaled importance (|coefficient × std dev|)")
plt.title("Feature importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

The features at the top of the chart have the greatest influence on predicted price. This gives the agency a clear, evidence-based picture of which property characteristics drive valuations. It also serves as a sanity check: if a feature we would expect to matter (like floor area) ranks low, or if an implausible feature ranks high, that is one of the warning signs the explainer described.

---

## 6. When to stop adding features

More features do not always mean a better model. Each additional feature:

- Adds complexity, making the model harder to explain to stakeholders.
- Risks introducing noise or multicollinearity (which we spotted in the correlation notebook).
- Must be available at prediction time. There is no point using a feature we will not have for new properties.

A practical approach:

1. Start with the features we expect to matter most (from domain knowledge and the correlation analysis).
2. Add features one at a time and check whether *test* R² improves.
3. Stop when adding more features stops helping.
4. If two models have similar test performance, prefer the simpler one.

In [ ]:
# Demonstrate incremental feature addition
feature_order = ["floor_area_sqm", "bedrooms", "distance_to_station_km",
                 "age_years", "garden_sqm", "bathrooms", "distance_to_school_km"]

incremental_results = []
for i in range(1, len(feature_order) + 1):
    selected = feature_order[:i]
    X_sel_train = X_train[selected]
    X_sel_test = X_test[selected]

    lr = LinearRegression().fit(X_sel_train, y_train)
    tr_r2 = lr.score(X_sel_train, y_train)
    te_r2 = lr.score(X_sel_test, y_test)

    incremental_results.append({
        "n_features": i,
        "added_feature": feature_order[i - 1],
        "train_r2": round(tr_r2, 4),
        "test_r2": round(te_r2, 4),
    })

pd.DataFrame(incremental_results)

Notice the pattern: the first few features produce large gains, and each subsequent feature adds less. At some point the improvement is marginal and not worth the added complexity. This is the diminishing-returns curve we should expect in any feature-selection process.

---

## 7. Final recommendation

Everything in this notebook has been building towards a practical question: what should the agency actually use? A good recommendation includes the model's test-set performance in terms the audience can act on.

In [ ]:
# Final model: use the full feature set
final_model = LinearRegression()
final_model.fit(X_train, y_train)

y_final_pred = final_model.predict(X_test)

final_mae = mean_absolute_error(y_test, y_final_pred)
final_rmse = root_mean_squared_error(y_test, y_final_pred)
final_r2 = r2_score(y_test, y_final_pred)

print("=" * 45)
print("FINAL MODEL EVALUATION (test set)")
print("=" * 45)
print(f"Features:  {len(features)}")
print(f"MAE:       £{final_mae:,.0f}")
print(f"RMSE:      £{final_rmse:,.0f}")
print(f"R²:        {final_r2:.3f}")
print("=" * 45)

The MAE is the number the agency cares about most: "on average, our valuations will be off by approximately £X." Whether that is acceptable depends on the business context. For a quick online estimate to attract sellers, it might be perfectly adequate. For a formal valuation that determines a mortgage offer, the agency would want to reduce the error further, perhaps by incorporating neighbourhood data, EPC ratings, or more advanced techniques.

---

## 8. Exercises

### Exercise 1: Evaluate a reduced model

Fit a model using only `floor_area_sqm` and `bedrooms`. Calculate MAE, RMSE, and R² on the test set. Compare these to the full model's test metrics. Is the full model worth the extra complexity?

In [ ]:
# Your code here


### Exercise 2: Residual analysis

Using the full model, create two plots side by side:
1. A scatter plot of residuals vs `floor_area_sqm` (test set only).
2. A scatter plot of residuals vs `age_years` (test set only).

Add a horizontal dashed line at y=0. In a comment, describe any patterns you see and what they might mean.

In [ ]:
# Your code here


### Exercise 3: Polynomial regression with multiple features

Using `floor_area_sqm` and `distance_to_station_km` (two features), fit polynomial regression models with degrees 1, 2, and 3. For each, record the training and test R². Display the results as a DataFrame. At which degree does overfitting start?

In [ ]:
# Your code here


### Exercise 4: Full evaluation report

Choose your best model from this notebook (justify your choice in a comment). Produce a full evaluation:
1. Print MAE, RMSE, and R² on the test set.
2. Plot residuals vs predicted values.
3. Plot a histogram of residuals.
4. Create a scatter plot of actual vs predicted prices with a 45-degree reference line (where actual = predicted).

Write a brief comment summarising whether the model is suitable for production use.

In [ ]:
# Your code here


---

## Summary

This notebook brought the full modelling workflow together. We now know how to build a model *and* how to honestly assess whether it works.

- The **train/test split** is the fundamental safeguard against overfitting. Test-set metrics are the numbers that matter.
- **MAE** and **RMSE** tell us the size of the errors in real units. **R²** tells us the proportion of variance explained.
- **Residual plots** reveal systematic problems that summary metrics hide.
- **Polynomial regression** demonstrated overfitting visually: training performance keeps improving while test performance degrades.
- **Feature importance** and **incremental feature addition** give us a principled way to decide how complex the model should be.

Across these three notebooks, we have moved from exploring correlations, to fitting regression lines and interpreting their coefficients, to evaluating whether those models can be trusted on new data. That sequence — explore, model, evaluate — is the core workflow for any predictive analysis.